# Cross-Source Searchlight Decoding: Accuracy Distribution

Load per-subject searchlight accuracy maps from the pkl file and visualize the distribution.

In [ ]:
import os
import numpy as np
import pickle
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import stats

In [ ]:
# ── Paths ──
output_dir = r'N:\Experimental_Data\yujunchen\projects\AI_IAPS\Decoding\searchlight_cross_source'
pkl_path = os.path.join(output_dir, 'searchlight_cross_source_results.pkl')

with open(pkl_path, 'rb') as f:
    all_results = pickle.load(f)

print("Keys in pkl:", list(all_results.keys()))
print()

searchlight_results = all_results['searchlight_results']
subs = all_results['subs']
cross_comparisons = all_results['cross_comparisons']
comp_keys = [f"train_{c1}v{c2}_test_{c3}v{c4}" for c1, c2, c3, c4 in cross_comparisons]

# Diagnostic: inspect one map
first_comp = comp_keys[0]
first_sub = subs[0]
sample = searchlight_results[first_comp][first_sub]
print(f"Sample map shape: {sample.shape}, dtype: {sample.dtype}")
print(f"  Total voxels: {sample.size}")
print(f"  NaN voxels: {np.isnan(sample).sum()}")
print(f"  Non-NaN voxels: {(~np.isnan(sample)).sum()}")
non_nan = sample[~np.isnan(sample)]
if len(non_nan) > 0:
    print(f"  Non-NaN range: [{non_nan.min():.4f}, {non_nan.max():.4f}]")
    print(f"  Non-NaN mean: {non_nan.mean():.4f}")
else:
    print("  WARNING: all values are NaN!")

## Compute Per-Subject Whole-Brain Mean Accuracy

In [ ]:
sub_means = {}

for comp in comp_keys:
    means = []
    for sub in subs:
        acc_map = searchlight_results[comp][sub]
        non_nan = acc_map[~np.isnan(acc_map)]
        if len(non_nan) > 0:
            means.append(non_nan.mean())
        else:
            means.append(np.nan)
    sub_means[comp] = np.array(means)
    valid_n = np.sum(~np.isnan(sub_means[comp]))
    print(f"{comp}:")
    print(f"  {valid_n}/{len(subs)} subjects, mean = {np.nanmean(sub_means[comp]):.4f}")

## Accuracy Distribution Plot

In [ ]:
short_labels = {
    'train_pleasantvneutral_test_pleasantAIvneutralAI':       'Nat P vs N\n\u2192 AI P vs N',
    'train_pleasantAIvneutralAI_test_pleasantvneutral':       'AI P vs N\n\u2192 Nat P vs N',
    'train_unpleasantvneutral_test_unpleasantAIvneutralAI':   'Nat U vs N\n\u2192 AI U vs N',
    'train_unpleasantAIvneutralAI_test_unpleasantvneutral':   'AI U vs N\n\u2192 Nat U vs N',
}

mpl.rcParams.update({
    'font.size': 11, 'font.family': 'Arial',
    'axes.linewidth': 1, 'xtick.major.width': 1, 'ytick.major.width': 1
})

fig, ax = plt.subplots(figsize=(7, 4.5))

positions = np.arange(len(comp_keys))
data = [sub_means[c] for c in comp_keys]
labels = [short_labels.get(c, c) for c in comp_keys]

parts = ax.violinplot(data, positions=positions,
                      showmeans=False, showmedians=False, showextrema=False)
for pc in parts['bodies']:
    pc.set_facecolor('#7FB3D8')
    pc.set_edgecolor('none')
    pc.set_alpha(0.35)

rng = np.random.default_rng(0)
for i, (pos, vals) in enumerate(zip(positions, data)):
    jitter = rng.uniform(-0.12, 0.12, size=len(vals))
    ax.scatter(pos + jitter, vals, s=18, color='#2166AC',
              alpha=0.7, edgecolors='white', linewidths=0.3, zorder=3)
    m = np.nanmean(vals)
    sem = np.nanstd(vals) / np.sqrt(np.sum(~np.isnan(vals)))
    ax.errorbar(pos + 0.25, m, yerr=sem, fmt='D', color='#B2182B',
                markersize=6, capsize=4, capthick=1.2, elinewidth=1.2, zorder=4)

ax.axhline(0.5, color='gray', ls='--', lw=1, zorder=1, label='Chance (50%)')
ax.set_xticks(positions)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Whole-brain mean decoding accuracy')
ax.set_title('Cross-Source Searchlight Decoding')
ax.legend(loc='upper right', frameon=False, fontsize=9)
ax.set_xlim(-0.5, len(comp_keys) - 0.5)
ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
plt.savefig(os.path.join(output_dir, 'cross_source_accuracy_distribution.png'),
            dpi=300, bbox_inches='tight')
plt.show()

## Summary Statistics

In [ ]:
print(f"{'Comparison':<60s} {'Mean':>7s} {'SEM':>7s} {'Min':>7s} {'Max':>7s}  {'t':>6s} {'p':>9s}")
print('-' * 106)
for c in comp_keys:
    vals = sub_means[c][~np.isnan(sub_means[c])]
    if len(vals) == 0:
        print(f"{c:<60s}   no valid data")
        continue
    m = vals.mean()
    sem = vals.std() / np.sqrt(len(vals))
    t_stat, p_val = stats.ttest_1samp(vals, 0.5)
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'n.s.'
    print(f"{c:<60s} {m:.4f}  {sem:.4f}  {vals.min():.4f}  {vals.max():.4f} {t_stat:>6.2f} {p_val:>9.1e} {sig}")

## Fix and Re-Save Per-Subject NIfTI Files

The original per-subject NIfTI files were saved with the mask header (likely uint8), which truncated all float accuracy values to 0. This cell re-saves them with a proper float32 header.

In [ ]:
# Re-save per-subject maps with correct float32 header
mask_file = r'N:\Experimental_Data\yujunchen\projects\data\masks\MNI152_T1_2mm_brain_mask.nii.gz'
beta_dir = r'N:\Experimental_Data\yujunchen\projects\AI_IAPS\GLM_singletrial\betas'

# Get the correct affine from the resampled mask
temp_nii = nib.load(os.path.join(beta_dir, 'Sub1', 'beta_0001.nii'))
from nilearn.image import resample_img
mask_img = nib.load(mask_file)
resampled_mask = resample_img(mask_img, target_affine=temp_nii.affine, target_shape=temp_nii.shape)
affine = resampled_mask.affine

sub_output_dir = os.path.join(output_dir, 'per_subject')
os.makedirs(sub_output_dir, exist_ok=True)

for comp in comp_keys:
    for sub in subs:
        acc_map_3d = searchlight_results[comp][sub]
        # Use a fresh header so float32 is preserved
        nii = nib.Nifti1Image(acc_map_3d.astype(np.float32), affine)
        nib.save(nii, os.path.join(sub_output_dir, f"{comp}_{sub}.nii.gz"))

# Also re-save group mean maps
for comp in comp_keys:
    all_maps = np.stack([searchlight_results[comp][sub] for sub in subs], axis=0)
    avg_map = np.nanmean(all_maps, axis=0).astype(np.float32)
    nib.save(nib.Nifti1Image(avg_map, affine),
             os.path.join(output_dir, f"{comp}_mean.nii.gz"))

print("Re-saved all NIfTI files with float32 header.")